In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [ ]:
data= pd.read_csv("Churn_Modelling.csv")
data.head()

In [ ]:
##extracting unimportant feature / preprocessing
data= data.drop(['RowNumber','CustomerId','Surname'], axis=1)
data

In [ ]:
## Label EEncoder of gender
label_encod_gender= LabelEncoder()
data['Gender']= label_encod_gender.fit_transform(data['Gender'])
data

In [ ]:
## OneHot Encoding ofGeography

from sklearn.preprocessing import OneHotEncoder
one_hot_geo= OneHotEncoder()
geo_encoder= one_hot_geo.fit_transform(data[['Geography']]).toarray()
geo_encoder

In [ ]:
one_hot_geo.get_feature_names_out(['Geography'])

In [ ]:
geo_encoder_df= pd.DataFrame(geo_encoder,columns= one_hot_geo.get_feature_names_out(['Geography']))
geo_encoder_df

In [ ]:
## Combine one hot encoder columns with the original data
data = data.drop('Geography', axis=1)          # pehle Geography hatao

data= pd.concat([data,geo_encoder_df],axis=1)
data.head()

In [ ]:
## Save the encoders and sscaler
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encod_gender,file)

with open('one_hot_geo.pkl','wb') as file:
    pickle.dump(one_hot_geo,file)

In [ ]:
data.head()

In [ ]:
## DiVide the dataset into indepent and dependent features
X= data.drop(['Exited'],axis=1)
y=data['Exited']

## Split the data in training and tetsing sets
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42)

## Scale these features
scale= StandardScaler()
X_train= scale.fit_transform(X_train)
X_test=scale.transform(X_test)

In [ ]:
data

In [ ]:
with open ('scale.pkl','wb') as file:
    pickle.dump(scale,file)

In [ ]:
data

**ANN Implementation**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [ ]:
(X_train.shape[1],)

In [ ]:
## Build Our ANN Model
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), ## HL1 Connected wwith input layer
    Dense(32,activation='relu'), ## HL2
    Dense(1,activation='sigmoid')  ## output layer
]

)

In [ ]:
model.summary()

In [ ]:
import tensorflow
opt= tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

In [ ]:
## model compilation
model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
#setup the tennsorboard
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [ ]:
## Set up Early Stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)


In [ ]:
### Train the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

In [ ]:
model.save('model.h5')

In [ ]:
##loading tensorboard extension
%load_ext tensorboard

In [ ]:
%tensorboard --logdir "logs/fit"